# Transaction Intelligence — Phase 3: fine-tune the teacher (Colab)

**Before you run:**
1. **Runtime → Change runtime type → GPU** (a free T4 is fine), then Save.
2. If you ran any cells earlier this session, also do **Runtime → Restart session** (resets PyTorch to Colab's version — important; a torch mismatch is what breaks the imports).

Then **Runtime → Run all**. The bar to beat (TF-IDF baseline): **gold subtype 0.77 / category 0.85**. When it finishes, send the printed `teacher/test` and `teacher/gold` numbers back to Claude.

In [ ]:
# 1) Get the code (idempotent — safe to re-run).
%cd /content
!rm -rf transaction-intelligence
!git clone https://github.com/thejayvaghela/transaction-intelligence.git
%cd transaction-intelligence
# PRIVATE repo? Replace the clone line above with these two:
#   from getpass import getpass; TOKEN = getpass('GitHub token: ')
#   !git clone https://{TOKEN}@github.com/thejayvaghela/transaction-intelligence.git

In [ ]:
# 2) Install deps WITHOUT touching Colab's PyTorch (upgrading torch breaks torchvision).
#    We install only what Colab lacks, then the package itself with --no-deps.
!pip install -q "transformers>=4.41" datasets accelerate sentencepiece mlflow-skinny
!pip install -q -e . --no-deps

In [ ]:
# 3) Recreate the dataset deterministically (data/ is gitignored; this rebuilds the exact splits).
!python scripts/build_dataset.py

In [ ]:
# 4) Train the teacher (DistilBERT, ~4 epochs; a few minutes on a T4).
#    Prints subtype/category macro-F1 on test + gold -- this is the result we want.
!python scripts/train_transformer.py

In [ ]:
# 5) (Optional) Download the trained model as a zip.
!zip -rq teacher.zip models/teacher
from google.colab import files
files.download('teacher.zip')

## After training
- Read the printed **`teacher/gold`** line and compare to the baseline (**0.77 / 0.85**).
- **Paste the `teacher/test` and `teacher/gold` lines back to Claude.**
- The model is in `models/teacher` (cell 5 downloads it as `teacher.zip` if you want it locally).

**To try the stronger DeBERTa-v3 teacher:** edit `configs/finetune.yaml` → `model_id: microsoft/deberta-v3-small`, then re-run the train cell.